In [2]:
pip install happybase


  Using cached happybase-1.3.0-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached ply-3.11-py2.py3-none-any.whl.metadata (844 bytes)
Using cached happybase-1.3.0-py2.py3-none-any.whl (26 kB)
   ---------------------------------------- 0.0/851.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/851.0 kB ? eta -:--:--
   ------------------------ --------------- 524.3/851.0 kB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 851.0/851.0 kB 3.4 MB/s  0:00:00
Using cached ply-3.11-py2.py3-none-any.whl (49 kB)
Using cached importlib_resources-7.1.0-py3-none-any.whl (37 kB)

   ---------------------------------------- 0/4 [ply]
   ---------------------------------------- 0/4 [ply]
   ---------- ----------------------------- 1/4 [thriftpy2]
   ---------- ----------------------------- 1/4 [thriftpy2]
   ---------- ----------------------------- 1/4 [thriftpy2]
   ---------- -------------

In [1]:
import happybase
import struct

# Connect to your Docker HBase container Thrift server
connection = happybase.Connection('localhost', port=9090)

SESS_TABLE = 'user_sessions'
PROD_TABLE = 'product_performance'

def setup_all_hbase_tables():
    print(" INITIALIZING HBASE TABLES ")
    existing_tables = connection.tables()
    
    # 1. Setup User Session Table
    if SESS_TABLE.encode() in existing_tables:
        connection.delete_table(SESS_TABLE, disable=True)
    connection.create_table(SESS_TABLE, {'session_meta': dict(), 'activity_stream': dict()})
    print(f"Created Table: '{SESS_TABLE}'")
    
    # 2. Setup Product Performance Table
    if PROD_TABLE.encode() in existing_tables:
        connection.delete_table(PROD_TABLE, disable=True)
    connection.create_table(PROD_TABLE, {'metrics': dict()})
    print(f"Created Table: '{PROD_TABLE}'\n")

def load_comprehensive_data():
    print(" INGESTING SESSIONS AND PRODUCT METRICS ")
    sess_table = connection.table(SESS_TABLE)
    prod_table = connection.table(PROD_TABLE)
    
    # Ingesting User Sessions (user_id + reverse_timestamp)
    # Formatted using: Max Timestamp Buffer (9999999999) - Epoch Timestamp
    sess_table.put(b'user_000042_8258149439', {
        b'session_meta:timestamp': b'2025-03-13T09:22:40',
        b'session_meta:duration': b'1450',
        b'session_meta:browser': b'Chrome',
        b'activity_stream:viewed_products': b'prod_00123,prod_03267'
    })
    sess_table.put(b'user_000042_8258204194', {
        b'session_meta:timestamp': b'2025-03-12T18:10:05',
        b'session_meta:duration': b'312',
        b'session_meta:browser': b'Safari',
        b'activity_stream:viewed_products': b'prod_00123'
    })
    print("[HBase] Loaded user session activity stream data.")

    # Ingesting Product Performance Over Time (product_id + date)
    # Using 8-byte big-endian integers to handle standard HBase high-speed counter increments
    prod_table.put(b'prod_00123_2025-03-12', {
        b'metrics:total_views': struct.pack('>q', 1540),
        b'metrics:add_to_cart_count': struct.pack('>q', 210)
    })
    prod_table.put(b'prod_00123_2025-03-13', {
        b'metrics:total_views': struct.pack('>q', 1890),
        b'metrics:add_to_cart_count': struct.pack('>q', 345)
    })
    print("[HBase] Loaded chronological product performance metrics.\n")

def execute_project_queries():
    print(" EXECUTING PROJECT DELIVERABLE QUERIES ")
    sess_table = connection.table(SESS_TABLE)
    prod_table = connection.table(PROD_TABLE)
    
    # Query 1: Retrieve user session data for a specific user
    print("\n Retrieving session data for target user: 'user_000042'")
    user_prefix = b"user_000042_"
    for key, data in sess_table.scan(row_prefix=user_prefix):
        print(f"  Row Key: {key.decode()} | "
              f"Time: {data[b'session_meta:timestamp'].decode()} | "
              f"Browser: {data[b'session_meta:browser'].decode()} | "
              f"Views: {data[b'activity_stream:viewed_products'].decode()}")

    # Query 2: Retrieve product performance timeline metrics
    print("\n Retrieving performance metrics for target product: 'prod_00123'")
    prod_prefix = b"prod_00123_"
    for key, data in prod_table.scan(row_prefix=prod_prefix):
        # Unpack the raw byte strings back into readable Python integers
        views = struct.unpack('>q', data[b'metrics:total_views'])[0]
        carts = struct.unpack('>q', data[b'metrics:add_to_cart_count'])[0]
        print(f"  Row Key: {key.decode()} | "
              f"Daily Views: {views} | "
              f"Cart Additions: {carts}")

if __name__ == "__main__":
    setup_all_hbase_tables()
    load_comprehensive_data()
    execute_project_queries()


 INITIALIZING HBASE TABLES 
Created Table: 'user_sessions'
Created Table: 'product_performance'

 INGESTING SESSIONS AND PRODUCT METRICS 
[HBase] Loaded user session activity stream data.
[HBase] Loaded chronological product performance metrics.

 EXECUTING PROJECT DELIVERABLE QUERIES 

 Retrieving session data for target user: 'user_000042'
  Row Key: user_000042_8258149439 | Time: 2025-03-13T09:22:40 | Browser: Chrome | Views: prod_00123,prod_03267
  Row Key: user_000042_8258204194 | Time: 2025-03-12T18:10:05 | Browser: Safari | Views: prod_00123

 Retrieving performance metrics for target product: 'prod_00123'
  Row Key: prod_00123_2025-03-12 | Daily Views: 1540 | Cart Additions: 210
  Row Key: prod_00123_2025-03-13 | Daily Views: 1890 | Cart Additions: 345
